# Q4: Feature Engineering

**Phase 5:** Feature Engineering & Aggregation  
**Points: 9 points**

**Focus:** Create derived features, perform time-based aggregations, calculate rolling windows.

**Lecture Reference:** Lecture 11, Notebook 2 ([`11/demo/02_wrangling_feature_engineering.ipynb`](https://github.com/christopherseaman/datasci_217/blob/main/11/demo/02_wrangling_feature_engineering.ipynb)), Phase 5. Also see Lecture 09 (rolling windows).

---

## Setup

In [80]:
# Import libraries
import pandas as pd
import numpy as np
import os

# Create output directory if it doesn't exist
os.makedirs('output', exist_ok=True)

# Load wrangled data from Q3
df = pd.read_csv("output/q3_wrangled_data.csv")
df["Measurement Timestamp"] = pd.to_datetime(df["Measurement Timestamp"])
df = df.set_index("Measurement Timestamp")

print(f"Loaded {len(df):,} records with datetime index")

Loaded 195,892 records with datetime index


## New Features (e.g., difference scores, categories)

In [81]:
# New features created
df["Temperature Difference"] = df["Air Temperature"] - df["Wet Bulb Temperature"]
df["Temp Ratio"] = df["Air Temperature"] / df["Wet Bulb Temperature"].replace(0, np.nan) # Avoid division by zero
df["Wind Speed Squared"] = df["Wind Speed"] ** 2
df["Air Temperature (F)"] = df["Air Temperature"] * 1.8 + 32
df["Comfort Index"] = (df["Air Temperature (F)"] + (df["Humidity"] / 100)) / 4

# Categorical: Air Temperature
bins_temp = [-22, 0, 19, 29, 39, float("inf")]
labels_temp = ["Freezing", "Cool", "Warm", "Hot", "Very Hot"]
df["Air Temperature Categories"] = pd.cut(
    df["Air Temperature"], bins=bins_temp, labels=labels_temp, right=True
).astype("category")

# Categorical: Wind Speed
bins_wind = [0, 4, 9, 19, 29, float("inf")]
labels_wind = ["Very Slow", "Slow", "Considerable", "Very Windy", "Extreme"]
df["Wind Speed Categories"] = pd.cut(
    df["Wind Speed"], bins=bins_wind, labels=labels_wind, right=True
).astype("category")

# Save Artifact 1
df_art1 = df.reset_index()
df_art1.to_csv("output/q4_features.csv", index=False)
print("✓ Saved output/q4_features.csv with shape", df_art1.shape)


✓ Saved output/q4_features.csv with shape (195892, 33)


## Rolling Variables

In [82]:
# Rolling window features
df = df.sort_index()

# Only predictor variables, avoid target variable (i.e., Air Temperature)
df['pressure_rolling_7h'] = df['Barometric Pressure'].rolling(window=7, min_periods=1).mean()
df['barometric_pressure_rolling_mean_7h'] = df['Barometric Pressure'].rolling('7h').mean()
df['solar_radiation_rolling_mean_7h'] = df['Solar Radiation'].rolling('7h').mean()
df['total_rain_rolling_mean_24h'] = df['Total Rain'].rolling('24h').mean()

# Save the updated DataFrame to a CSV file
output_filename = 'output/q4_data_with_rolling_features.csv'
df.to_csv(output_filename, index=False) 

print(f"DataFrame successfully saved to {output_filename}")

DataFrame successfully saved to output/q4_data_with_rolling_features.csv


In [83]:
new_features = [
    "Temperature Difference",
    "Temp Ratio",
    "Wind Speed Squared",
    "Air Temperature (F)",
    "Comfort Index",
    "Air Temperature Categories",
    "Wind Speed Categories",
    "pressure_rolling_7h",
    "barometric_pressure_rolling_mean_7h",
    "solar_radiation_rolling_mean_7h",
    "total_rain_rolling_mean_24h"
]

feature_list = list(dict.fromkeys(new_features))

with open("output/q4_feature_list.txt", "w") as f:
    for name in feature_list:
        f.write(name + "\n")

print("✓ Saved output/q4_feature_list.txt with", len(feature_list), "features")
print("Features:", feature_list)

✓ Saved output/q4_feature_list.txt with 11 features
Features: ['Temperature Difference', 'Temp Ratio', 'Wind Speed Squared', 'Air Temperature (F)', 'Comfort Index', 'Air Temperature Categories', 'Wind Speed Categories', 'pressure_rolling_7h', 'barometric_pressure_rolling_mean_7h', 'solar_radiation_rolling_mean_7h', 'total_rain_rolling_mean_24h']
